# Insturctions to follow before executing this script

## Step 1: Create the Isolated phase1_ltx (Run in Terminal)

Before launching Jupyter, open your compute node terminal and run these commands to create a pristine environment for LTX-2.3. Note that we split the installation commands to prevent flashpack from overwriting our specific CUDA version of PyTorch, and we pin huggingface_hub<1.0 to prevent version conflicts with Transformers.

### Deactivate the old environment
conda deactivate
conda remove --name phase1_ltx --all

### Create a fresh Python 3.11 environment for LTX
conda create -n phase1_ltx python=3.11 -y
conda activate phase1_ltx

#### 1. Install PyTorch (highest available stable version for cu121)
pip install torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

#### 2. Install flashpack WITH NO DEPENDENCIES (prevents it from upgrading PyTorch)
pip install flashpack==0.1.2 --no-deps

#### 3. Install the rest of the LTX dependencies (pinning HF Hub for Transformers compatibility)
pip install transformers==4.57.6 accelerate einops scipy av scikit-image>=0.25.2
pip install gradio spaces "huggingface_hub<1.0"

### Launch Jupyter in this new environment on port 8905
jupyter notebook --ip=0.0.0.0 --port=8905 --allow-root --no-browser &


## Step 2: Establish the SSH Tunnels

You need to tunnel both the Jupyter port (8905) and the Gradio port (7860) back to your laptop. Run this on your LOCAL laptop's terminal:

### Tunnel for Jupyter
ssh -f -N -L 5402:scn30-mn:8888 rahkus@login.npsf.cdac.in
### Tunnel for Gradio
ssh -f -N -L 7860:scn30-mn:7860 rahkus@login.npsf.cdac.in


In [1]:
import core_config

print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")

[CORE] Secrets & Config loaded successfully from /nlsasfs/home/lmedge/rahkus/.env
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/files
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/info
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.jupyter/lab/workspaces
[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.


In [ ]:
# Read-only process inspection; never kill a hard-coded PID on a shared host.
import os, subprocess
user = os.environ.get('USER', '')
if user:
    result = subprocess.run(['ps', '-u', user, '-o', 'pid,lstart,etime,comm'], capture_output=True, text=True, check=False)
    print(result.stdout if result.returncode == 0 else result.stderr)
else:
    print('USER is not set; run a read-only process inspection command manually.')


In [3]:
!nvidia-smi

Mon Aug  3 19:26:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   27C    P0             53W /  400W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# Explicit runtime validation only.
# Heavy installs/clones are intentionally NOT performed inside the inference notebook.
from pathlib import Path
import importlib.util
import os
import sys

LTX_REPO_URL = "https://github.com/Lightricks/LTX-2.git"
LTX_COMMIT_SHA = "ae855f8538843825f9015a419cf4ba5edaf5eec2"
LTX_REPO_DIR = Path(
    os.environ.get(
        "AIEROS_LTX_REPO_DIR",
        Path(core_config.SYSTEM_DIRS["cache"]) / "third_party" / "LTX-2",
    )
)
LTX_CORE_DIR = LTX_REPO_DIR / "packages" / "ltx-core"
LTX_PIPELINE_DIR = LTX_REPO_DIR / "packages" / "ltx-pipelines"

for required in (LTX_CORE_DIR, LTX_PIPELINE_DIR):
    if not required.is_dir():
        raise RuntimeError(
            f"Missing prepared LTX runtime component: {required}. "
            "Prepare the pinned runtime explicitly before launching inference."
        )

if importlib.util.find_spec("xformers") is None:
    raise RuntimeError(
        "xformers is not installed in the active environment. "
        "Install runtime dependencies during environment provisioning, not from this notebook."
    )

sys.path.insert(0, str(LTX_PIPELINE_DIR / "src"))
sys.path.insert(0, str(LTX_CORE_DIR / "src"))

print(f"LTX runtime checkout: {LTX_REPO_DIR}")
print(f"Pinned commit: {LTX_COMMIT_SHA}")
print("Runtime validation complete; no package install or repository mutation performed.")


Installing xformers (v0.0.28.post3 for Torch 2.5.1)...
Installing ltx-core and ltx-pipelines from cloned repo...
Obtaining file:///nlsasfs/home/lmedge/rahkus/LTX-2/packages/ltx-core
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
Obtaining file:///nlsasfs/home/lmedge/rahkus/LTX-2/packages/ltx-pipelines
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Ge

In [8]:
import os
import sys
import subprocess
import logging
import random
import time
import glob
from pathlib import Path
from datetime import datetime
import json
import struct
import torch
import multiprocessing as mp
import gradio as gr
import numpy as np
from storage_manager import managed_storage_operation
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes

# ==============================================================================
# 1. CORE SYSTEM & MODE CONFIGURATION
# ==============================================================================

# GLOBAL TOGGLE: "GRADIO", "WORKER_SINGLE", or "WORKER_PARALLEL"
RUN_MODE = "GRADIO"  
AVAILABLE_GPUS = ["0", "1", "2", "3"] # Used only for WORKER_PARALLEL

# Prevent VRAM fragmentation on 40GB cards
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_USE_XET"] = "0"
os.environ["no_proxy"] = "localhost, 127.0.0.1, ::1, 0.0.0.0"
os.environ["NO_PROXY"] = "localhost, 127.0.0.1, ::1, 0.0.0.0"

MODEL_TYPE = "ltx-2.3-22b"
OUTPUT_TYPE_VIDEO = "video"
OUTPUT_TYPE_IMAGE = "image"
MAX_SEED = np.iinfo(np.int32).max
DEFAULT_FRAME_RATE = 24.0

RESOLUTIONS = {
    "high": {"16:9": (1536, 1024), "9:16": (1024, 1536), "1:1": (1024, 1024)},
    "low": {"16:9": (768, 512), "9:16": (512, 768), "1:1": (768, 768)},
}

AIEROS_OUTPUT_DIR = Path(core_config.SYSTEM_DIRS["outputs"]) / "videos"
AIEROS_INPUT_DIR = Path(core_config.SYSTEM_DIRS["inputs"]) / "images"
JOB_DIR = Path(core_config.SYSTEM_DIRS["inputs"]) / "jobs"
AIEROS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AIEROS_INPUT_DIR.mkdir(parents=True, exist_ok=True)
JOB_DIR.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# 2. UTILITIES & LOGGING
# ==============================================================================
class DualLogger:
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log_file = open(filepath, "a", buffering=1)
    def write(self, message):
        self.terminal.write(message)
        self.log_file.write(message)
        self.terminal.flush()
        self.log_file.flush()
    def flush(self):
        self.terminal.flush()
        self.log_file.flush()

def get_best_gpu():
    try:
        smi_output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,nounits,noheader"], 
            encoding="utf-8"
        )
        memory_free = [int(x.strip()) for x in smi_output.strip().split('\n')]
        best_gpu = memory_free.index(max(memory_free))
        print(f"[SYSTEM] Auto-selected GPU {best_gpu} ({memory_free[best_gpu]} MB free)")
        return str(best_gpu)
    except Exception as e:
        raise RuntimeError(f"Could not determine available GPU memory via nvidia-smi: {e}") from e

# ==============================================================================
# 3. HEAVY ML LOGIC (Isolated for Multiprocessing)
# ==============================================================================
def apply_fuse_patches():
    from ltx_core.loader.primitives import StateDict
    from ltx_core.loader.sft_loader import SafetensorsStateDictLoader
    from ltx_core.model.transformer import attention as _attn_mod
    
    try:
        from xformers.ops import memory_efficient_attention as _mea
        _attn_mod.memory_efficient_attention = _mea
        print("[ATTN] memory_efficient_attention patched successfully.")
    except Exception as e:
        print(f"[ATTN] xformers patch FAILED: {e}")

    try:
        from xformers.ops.fmha import _set_use_fa3
        _set_use_fa3(False)
        print("[ATTN] xformers FA3 dispatch disabled (Blackwell/Hopper compatibility)")
    except Exception:
        pass

    _SAFETENSORS_DTYPE_MAP = {
        "F64": torch.float64, "F32": torch.float32, "F16": torch.float16,
        "BF16": torch.bfloat16, "F8_E5M2": torch.float8_e5m2, "F8_E4M3": torch.float8_e4m3fn,
        "I64": torch.int64, "I32": torch.int32, "I16": torch.int16,
        "I8": torch.int8, "U8": torch.uint8, "BOOL": torch.bool,
    }

    def _patched_load(self, path, sd_ops, device=None):
        sd = {}
        size = 0
        dtype = set()
        device = device or torch.device("cpu")
        model_paths = path if isinstance(path, list) else [path]
        for shard_path in model_paths:
            with open(shard_path, "rb") as f:
                header_len = struct.unpack("<Q", f.read(8))[0]
                header = json.loads(f.read(header_len).decode("utf-8"))
                data_base = 8 + header_len
                for name, meta in header.items():
                    if name == "__metadata__": continue
                    expected_name = name if sd_ops is None else sd_ops.apply_to_key(name)
                    if expected_name is None: continue
                    start, end = meta["data_offsets"]
                    f.seek(data_base + start)
                    buf = f.read(end - start)
                    t = torch.frombuffer(bytearray(buf), dtype=_SAFETENSORS_DTYPE_MAP[meta["dtype"]]).reshape(meta["shape"])
                    t = t.to(device=device, non_blocking=True, copy=False)
                    kvs = (((expected_name, t),) if sd_ops is None else sd_ops.apply_to_key_value(expected_name, t))
                    for key, v in kvs:
                        size += v.nbytes
                        dtype.add(v.dtype)
                        sd[key] = v
        return StateDict(sd=sd, device=device, size=size, dtype=dtype)

    SafetensorsStateDictLoader.load = _patched_load
    print("[FUSE-PATCH] SafetensorsStateDictLoader.load replaced (chunked-read)")

def load_pipeline_instance(gpu_id):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    apply_fuse_patches()

    from huggingface_hub import hf_hub_download, snapshot_download
    from ltx_core.quantization import QuantizationPolicy
    from ltx_pipelines.distilled import DistilledPipeline

    print(f"[GPU {gpu_id}] Verifying LTX models in local cache...")
    try:
        distilled_checkpoint_path = hf_hub_download(
            repo_id="Lightricks/LTX-2.3",
            filename="ltx-2.3-22b-distilled-1.1.safetensors",
            local_files_only=True,
        )
        spatial_upsampler_path = hf_hub_download(
            repo_id="Lightricks/LTX-2.3",
            filename="ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
            local_files_only=True,
        )
        gemma_root = snapshot_download(
            repo_id="google/gemma-3-12b-it-qat-q4_0-unquantized",
            local_files_only=True,
        )
        print(f"[GPU {gpu_id}] Cache hit; no model download required.")
    except Exception:
        required_bytes = sum(
            [
                estimate_hf_repo_bytes(
                    "Lightricks/LTX-2.3",
                    filenames=[
                        "ltx-2.3-22b-distilled-1.1.safetensors",
                        "ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
                    ],
                ),
                estimate_hf_repo_bytes(
                    "google/gemma-3-12b-it-qat-q4_0-unquantized"
                ),
            ]
        )
        required_gb = (
            required_bytes / BYTES_PER_GB
            + max(
                0.0,
                float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")),
            )
        )
        print(
            f"[GPU {gpu_id}] Cache incomplete. "
            f"Storage admission budget: {required_gb:.2f} GB"
        )

        with managed_storage_operation(
            "models",
            required_gb=required_gb,
            require_fresh_quota=True,
            minimum_free_gb=float(
                os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0")
            ),
        ):
            distilled_checkpoint_path = hf_hub_download(
                repo_id="Lightricks/LTX-2.3",
                filename="ltx-2.3-22b-distilled-1.1.safetensors",
            )
            spatial_upsampler_path = hf_hub_download(
                repo_id="Lightricks/LTX-2.3",
                filename="ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
            )
            gemma_root = snapshot_download(
                repo_id="google/gemma-3-12b-it-qat-q4_0-unquantized"
            )

    print(f"[GPU {gpu_id}] Loading DistilledPipeline...")
    pipeline = DistilledPipeline(
        distilled_checkpoint_path=distilled_checkpoint_path,
        spatial_upsampler_path=spatial_upsampler_path,
        gemma_root=gemma_root,
        loras=[],
        quantization=QuantizationPolicy.fp8_cast(),
    )
    print(f"[GPU {gpu_id}] Pipeline ready!")
    return pipeline

@torch.inference_mode()
def generate_asset(pipeline, target_gpu_id, input_image, prompt: str, duration: float, enhance_prompt: bool = True, seed: int = 42, randomize_seed: bool = True, height: int = 1024, width: int = 1536):
    from ltx_core.model.video_vae import TilingConfig, get_video_chunks_number
    from ltx_pipelines.utils.args import ImageConditioningInput
    from ltx_pipelines.utils.media_io import encode_video
    
    try:
        width = int(round(width / 64.0)) * 64
        height = int(round(height / 64.0)) * 64
        
        torch.cuda.reset_peak_memory_stats()
        current_seed = random.randint(0, MAX_SEED) if randomize_seed else int(seed)
        num_frames = ((int(duration * DEFAULT_FRAME_RATE) - 1 + 7) // 8) * 8 + 1
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        base_filename = f"{timestamp}_{OUTPUT_TYPE_VIDEO}_{MODEL_TYPE}"
        print(f"[GPU {target_gpu_id}] Generating: {width}x{height}, {num_frames} frames ({duration}s), seed={current_seed}")
        
        images = []
        if input_image is not None:
            input_filename = f"{timestamp}_{OUTPUT_TYPE_IMAGE}_input_{MODEL_TYPE}.jpg"
            temp_image_path = AIEROS_INPUT_DIR / input_filename
            if hasattr(input_image, "save"): 
                input_image.save(temp_image_path)
            else: 
                temp_image_path = Path(input_image)
            images = [ImageConditioningInput(path=str(temp_image_path), frame_idx=0, strength=1.0)]

        tiling_config = TilingConfig.default()
        video_chunks_number = get_video_chunks_number(num_frames, tiling_config)

        video, audio = pipeline(
            prompt=prompt, seed=current_seed, height=height, width=width,
            num_frames=num_frames, frame_rate=DEFAULT_FRAME_RATE, images=images,
            tiling_config=tiling_config, enhance_prompt=enhance_prompt,
        )

        output_video_path = AIEROS_OUTPUT_DIR / f"{base_filename}.mp4"
        output_json_path = AIEROS_OUTPUT_DIR / f"{base_filename}_info.json"

        output_budget_gb = max(
            0.05,
            float(duration) * 100 * 1_000_000 / 8 / BYTES_PER_GB * 2.0,
        )

        with managed_storage_operation(
            "outputs",
            required_gb=output_budget_gb,
            require_fresh_quota=False,
            minimum_free_gb=0.05,
        ):
            encode_video(
                video=video,
                fps=DEFAULT_FRAME_RATE,
                audio=audio,
                output_path=str(output_video_path),
                video_chunks_number=video_chunks_number,
            )

            metadata = {
            "timestamp": timestamp,
            "model_type": MODEL_TYPE,
            "output_type": OUTPUT_TYPE_VIDEO,
            "file_name": f"{base_filename}.mp4",
            "generation_parameters": {
                "prompt": prompt, "enhance_prompt": enhance_prompt, "duration_seconds": duration,
                "frame_rate": DEFAULT_FRAME_RATE, "total_frames": num_frames,
                "resolution": {"width": width, "height": height}, "seed": current_seed,
                "randomize_seed": randomize_seed, "image_conditioned": input_image is not None,
                "target_gpu_id": target_gpu_id
            }
        }
            with open(output_json_path, "w", encoding="utf-8") as json_file:
                json.dump(metadata, json_file, indent=4)

        print(f"[GPU {target_gpu_id}] Saved Video to: {output_video_path}")
        return str(output_video_path), str(output_json_path), current_seed
    
    except Exception as e:
        import traceback
        print(f"[GPU {target_gpu_id}] Error: {str(e)}\n{traceback.format_exc()}")
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        print(f"[SYSTEM] Generation failed. Dumped VRAM cache to prevent hoarding on GPU {target_gpu_id}.")
        return None, None, current_seed

# ==============================================================================
# 4. WORKER PROCESS LOGIC
# ==============================================================================
def worker_process(gpu_id):
    log_path = JOB_DIR / f"worker_GPU{gpu_id}.log"
    sys.stdout = DualLogger(str(log_path))
    sys.stderr = sys.stdout

    try:
        from hf_upload_daemon import AssetQueue
        queue_manager = AssetQueue()
    except ImportError:
        print(f"[GPU {gpu_id}] hf_upload_daemon missing. Skiping queue submission.")
        queue_manager = None

    pipeline = load_pipeline_instance(gpu_id)
    print(f"\n[GPU {gpu_id}] Worker Mode Active. Polling for JSON jobs...")

    active_processing_file = None

    while True:
        try:
            job_files = glob.glob(str(JOB_DIR / "*.json"))
            job_files = [f for f in job_files if f.endswith(".json")]
            
            if not job_files:
                time.sleep(5)
                continue
                
            target_job = job_files[0]
            active_processing_file = target_job + ".processing"
            
            # ATOMIC LOCK: Prevents multiple GPUs from grabbing the same job
            try:
                os.rename(target_job, active_processing_file)
            except FileNotFoundError:
                active_processing_file = None
                continue
            except OSError:
                time.sleep(1)
                continue

            with open(active_processing_file, 'r') as f:
                job = json.load(f)
                
            print(f"\n[GPU {gpu_id}] Processing Job: {os.path.basename(target_job)}")
            
            prompt = job.get("prompt", "Cinematic landscape, 4k, professional cinematography")
            image_path = job.get("image_path", None)
            duration = job.get("duration", 3.0) 
            width = job.get("width", 1536)
            height = job.get("height", 1024) 
            enhance = job.get("enhance_prompt", False)
            num_takes = job.get("num_takes", 1) 
            
            success_count = 0
            for take in range(num_takes):
                print(f"[GPU {gpu_id}] -> Take {take + 1}/{num_takes}...")
                vid_path, json_path, final_seed = generate_asset(
                    pipeline, gpu_id, image_path, prompt, duration, enhance, 0, True, height, width
                )
                
                if vid_path and json_path:
                    success_count += 1
                    if queue_manager: queue_manager.submit(asset_path=vid_path, json_path=json_path)
                elif not vid_path:
                    print(f"[GPU {gpu_id}] -> Take {take + 1} failed. Skipping to next.")
            
            # FAIL-SAFE: Only delete the job if it successfully generated at least 1 video.
            if success_count > 0:
                os.remove(active_processing_file)
                print(f"[GPU {gpu_id}] Job completed ({success_count}/{num_takes} successful)")
            else:
                failed_path = active_processing_file.replace(".processing", ".failed")
                os.rename(active_processing_file, failed_path)
                print(f"[GPU {gpu_id}] FATAL: All takes failed. Job renamed to .failed")
                
            active_processing_file = None

        except KeyboardInterrupt:
            if active_processing_file and os.path.exists(active_processing_file):
                os.rename(active_processing_file, active_processing_file.replace(".processing", ""))
            break
        except Exception as e:
            print(f"[GPU {gpu_id}] Error processing job: {e}")
            if active_processing_file and os.path.exists(active_processing_file):
                os.rename(active_processing_file, active_processing_file.replace(".processing", ".error"))
            active_processing_file = None
            time.sleep(5)

# ==============================================================================
# 5. MASTER EXECUTION & ROUTING
# ==============================================================================
if __name__ == "__main__":
    
    # Startup Sweep: Clear orphaned jobs from previous crashes
    orphans = glob.glob(str(JOB_DIR / "*.processing"))
    for orphan in orphans:
        try:
            os.rename(orphan, orphan.replace(".processing", ""))
            print(f"[SYSTEM] Swept orphaned file back to queue: {os.path.basename(orphan)}")
        except Exception:
            pass

    if RUN_MODE == "GRADIO":
        target = get_best_gpu()
        print(f"\n[SYSTEM] Launching Gradio UI on GPU {target}...")
        pipeline = load_pipeline_instance(target)
        
        def detect_aspect_ratio(image) -> str:
            if image is None: return "16:9"
            w, h = image.size if hasattr(image, "size") else image.shape[:2]
            ratio = w / h
            candidates = {"16:9": 16 / 9, "9:16": 9 / 16, "1:1": 1.0}
            return min(candidates, key=lambda k: abs(ratio - candidates[k]))

        def on_image_upload(image, high_res):
            aspect = detect_aspect_ratio(image)
            tier = "high" if high_res else "low"
            w, h = RESOLUTIONS[tier][aspect]
            return gr.update(value=w), gr.update(value=h)

        def on_highres_toggle(image, high_res):
            aspect = detect_aspect_ratio(image)
            tier = "high" if high_res else "low"
            w, h = RESOLUTIONS[tier][aspect]
            return gr.update(value=w), gr.update(value=h)

        def gradio_wrapper(*args):
            vid, json_path, seed = generate_asset(pipeline, target, *args)
            return vid, seed

        with gr.Blocks(title="LTX-2.3 Distilled") as demo:
            gr.Markdown("# LTX-2.3 Distilled (22B): Interactive Testing Interface")
            with gr.Row():
                with gr.Column():
                    input_image = gr.Image(label="Input Image (Optional)", type="pil")
                    prompt = gr.Textbox(label="Prompt", value="Arri Alexa footage, 35mm film grain. A tight macro shot of glowing bioluminescent moss on a wet rock, extremely shallow depth of field, f/2.8, smooth cinematic pan.", lines=3)
                    with gr.Row():
                        duration = gr.Slider(label="Duration (seconds)", minimum=1.0, maximum=10.0, value=3.0, step=0.1)
                        with gr.Column():
                            enhance_prompt = gr.Checkbox(label="Enhance Prompt", value=False)
                            high_res = gr.Checkbox(label="High Resolution", value=True)
                    generate_btn = gr.Button("Generate Video", variant="primary", size="lg")
                    with gr.Accordion("Advanced Settings", open=False):
                        seed = gr.Slider(label="Seed", minimum=0, maximum=MAX_SEED, value=10, step=1)
                        randomize_seed = gr.Checkbox(label="Randomize Seed", value=True)
                        with gr.Row():
                            width = gr.Number(label="Width", value=1536, precision=0)
                            height = gr.Number(label="Height", value=1024, precision=0)
                with gr.Column():
                    output_video = gr.Video(label="Generated Video", autoplay=True)

            input_image.change(fn=on_image_upload, inputs=[input_image, high_res], outputs=[width, height])
            high_res.change(fn=on_highres_toggle, inputs=[input_image, high_res], outputs=[width, height])
            generate_btn.click(
                fn=gradio_wrapper,
                inputs=[input_image, prompt, duration, enhance_prompt, seed, randomize_seed, height, width],
                outputs=[output_video, seed],
            )

        demo.launch(server_name="0.0.0.0", server_port=7860, inline=True, share=False)

    else:
        # ---------------------------------------------------------
        # MULTIPROCESSING WORKER MODE
        # Requires 'spawn' method in Jupyter to prevent CUDA crashes
        # ---------------------------------------------------------
        try:
            mp.set_start_method('spawn', force=True)
        except RuntimeError:
            pass 

        processes = []
        try:
            if RUN_MODE == "WORKER_PARALLEL":
                print(f"[SYSTEM] Launching {len(AVAILABLE_GPUS)} concurrent GPU workers...")
                for gpu in AVAILABLE_GPUS:
                    p = mp.Process(target=worker_process, args=(gpu,))
                    p.start()
                    processes.append(p)
                    time.sleep(2) # Stagger boot times to prevent disk I/O spikes
                    
            elif RUN_MODE == "WORKER_SINGLE":
                target = get_best_gpu()
                print(f"[SYSTEM] Launching SINGLE worker on GPU {target}...")
                p = mp.Process(target=worker_process, args=(target,))
                p.start()
                processes.append(p)

            # Master Loop: Keeps the main cell alive while background processes run
            while True:
                time.sleep(1)
                if all(not p.is_alive() for p in processes):
                    break

        except KeyboardInterrupt:
            print("\n[SYSTEM] Master Shutdown Triggered. Broadcasting stop signals...")
            for p in processes:
                p.join(timeout=10)
                if p.is_alive():
                    p.terminate()
            print("[SYSTEM] All GPUs successfully shut down.")

[SYSTEM] Auto-selected GPU 0 (40323 MB free)

[SYSTEM] Launching Gradio UI on GPU 0...
[ATTN] memory_efficient_attention patched successfully.
[ATTN] xformers FA3 dispatch disabled (Blackwell/Hopper compatibility)
[FUSE-PATCH] SafetensorsStateDictLoader.load replaced (chunked-read)
[GPU 0] Downloading / Verifying LTX Models...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

[GPU 0] Loading DistilledPipeline...
[GPU 0] Pipeline ready!
* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


In [6]:
# demo.close()

In [7]:
!nvidia-smi

Mon Aug  3 19:26:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   26C    P0             53W /  400W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----